In [1]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the capital of Bulgaria?"}
    ]
)

response.content[0].text 

'The capital of Bulgaria is **Sofia**. It is the largest city in the country and serves as the political, economic, and cultural center of Bulgaria.'

In [2]:

calculator_tool_definition = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Perform basic arithmetic operations.",
        "parameters": {
            "type": "object",
            "properties": {
                "operator": {
                    "type": "string",
                    "description": "Arithmetic operation to perform",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "first_number": {
                    "type": "number",
                    "description": "First number for the calculation"
                },
                "second_number": {
                    "type": "number",
                    "description": "Second number for the calculation"
                }
            },
            "required": ["operator", "first_number", "second_number"],
        }
    }
}
   

In [3]:
def calculator(operator: str, first_number: float, second_number: float):
    if operator == 'add':
        return first_number + second_number
    elif operator == 'subtract':
        return first_number - second_number
    elif operator == 'multiply':
        return first_number * second_number
    elif operator == 'divide':
        if second_number == 0:
            raise ValueError("Cannot divide by zero")
        return first_number / second_number
    else:
        raise ValueError(f"Unsupported operator: {operator}")

In [4]:
import json
from litellm import completion

"""
This example demonstrates how to define a tool for basic arithmetic operations and how to use it in a conversation with the LLM.
The tool is defined as a function with a specific schema, and the LLM can call this tool when it recognizes that a calculation is needed.
"""
tools = [calculator_tool_definition]

response_without_tool = completion(
    model='claude-sonnet-4-6',  
    messages=[{"role": "user", "content": "What is the capital of South Korea?"}],
    tools=tools,
)
print("# Capital question doesn't need a tool call")
print(response_without_tool.choices[0].message.content)
print(response_without_tool.choices[0].message.tool_calls)

response_with_tool = completion(
    model='claude-sonnet-4-6',
    messages=[{"role": "user", "content": "What is 1234 x 5678?"}],
    tools=tools,
)
print("\n# Multiplication question needs a tool call")
print(response_with_tool.choices[0].message.content)
print(response_with_tool.choices[0].message.tool_calls)

# Capital question doesn't need a tool call
The capital of South Korea is **Seoul**. It is the largest city in South Korea and serves as the country's political, economic, and cultural center. Seoul has been the capital since the Joseon Dynasty, founded in 1392, making it one of the longest-serving capitals in the world.
None

# Multiplication question needs a tool call
None
[ChatCompletionMessageToolCall(index=0, caller={'type': 'direct'}, function=Function(arguments='{"operator": "multiply", "first_number": 1234, "second_number": 5678}', name='calculator'), id='toolu_01GTYLH7Rd7SfdaGbe9JSF2a', type='function')]


In [ ]:
"""Execute the tool call if present in the response."""
ai_message = response_with_tool.choices[0].message
if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        if function_name == "calculator":
            result = calculator(**function_args)
            print(f"{function_name}({function_args}) -> {result}")


messages = [{"role": "user", "content": "What is 1234 x 5678?"}]
ai_message = response_with_tool.choices[0].message
messages.append({                                # A: append the assistant's tool call
    "role": "assistant",
    "content": ai_message.content,
    "tool_calls": ai_message.tool_calls,
})
if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name  # B: parse + execute the tool
        function_args = json.loads(tool_call.function.arguments)
        if function_name == "calculator":
            result = calculator(**function_args)
            messages.append({                    # C: append the tool result
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result),
            })

final_response = completion(
    model='claude-sonnet-4-6',
    messages=messages,
)
print("Messages:", messages)
print("Final Answer:", final_response.choices[0].message.content)

calculator({'operator': 'multiply', 'first_number': 1234, 'second_number': 5678}) -> 7006652

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: OpenAIException - Your API key has expired. Create a new API key to continue.